# NL_SQL autotune — Colab SERVE (tuned student behind a public URL)

Runs vLLM with the QLoRA adapter on a single T4 and exposes it via a cloudflared
quick tunnel, so the Windows-side harness can eval `sqltuned` and the base model.

Why bnb-4bit: fp16 7B (~15GB) does not fit a 16GB T4; the adapter was trained on
`unsloth/qwen2.5-coder-7b-instruct-bnb-4bit`, so 4-bit serving matches the
training-time base exactly.

Steps: Runtime -> Change runtime type -> **T4 GPU** -> Run all. Two prompts:
Drive authorisation, and (first time only) `kaggle.json` upload
(Kaggle -> Settings -> Create New API Token). Then leave the tab open.
The tunnel URL is printed below the last cell AND pushed to the private
Kaggle dataset `liovinajo/nlsql-tunnel` for the Windows side to poll.


In [ ]:
BASE_MODEL = "unsloth/qwen2.5-coder-7b-instruct-bnb-4bit"  # training-time base, pre-quantized
ADAPTER_DATASET = "liovinajo/nlsql-adapter"

DRIVE_DIR = "/content/drive/MyDrive/nlsql_autotune"  # kaggle.json persists here
ADAPTER_LOCAL = "/content/nlsql_adapter"
SERVE_HOURS = 8.0
PORT = 8000

In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive, files

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, flush=True)

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

# kaggle.json: reuse the copy on Drive if a previous session left one, else ask once.
cred = Path("/root/.kaggle/kaggle.json")
cred.parent.mkdir(parents=True, exist_ok=True)
drive_cred = Path(DRIVE_DIR) / "kaggle.json"
if drive_cred.exists():
    cred.write_bytes(drive_cred.read_bytes())
    print("kaggle.json taken from Drive", flush=True)
else:
    print("upload kaggle.json (Kaggle -> Settings -> Create New API Token)", flush=True)
    up = files.upload()
    cred.write_bytes(next(iter(up.values())))
    drive_cred.write_bytes(cred.read_bytes())
cred.chmod(0o600)

In [ ]:
import shutil
import subprocess
from pathlib import Path

dst = Path(ADAPTER_LOCAL)
if dst.exists():
    shutil.rmtree(dst)
dst.mkdir(parents=True)
subprocess.run(
    ["kaggle", "datasets", "download", ADAPTER_DATASET, "-p", str(dst), "--unzip"],
    check=True,
)
# Kaggle unpacks uploaded zips flat (v10 lesson) -- self-home on the config.
hits = sorted(dst.rglob("adapter_config.json"))
if not hits:
    raise RuntimeError(f"no adapter_config.json under {dst} -- wrong dataset?")
adapter_dir = str(hits[0].parent)
if not Path(adapter_dir, "adapter_model.safetensors").exists():
    raise RuntimeError(f"adapter_model.safetensors missing next to {hits[0]}")
print(f"adapter_dir resolved: {adapter_dir}", flush=True)

In [ ]:
import json
import re
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

# DEAD PATH (2026-07-21): vLLM does not run on the Colab image at all — cu13
# wheel linkage, an aimv2 collision on 0.9.2, and triton PassManager failures
# on SM 7.5 LoRA kernels. Serving now goes through the transformers+peft+bnb
# shim instead (plan_autotune.md). Kept for the record; do not retry.
# Colab ships torch 2.11+cu128; the latest vLLM PyPI wheel is built for CUDA 13
# and dies on `libcudart.so.13`. Pin vLLM to the cu128 build via uv's torch-backend
# selector so it matches the preinstalled torch instead of the driver's CUDA 13.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(
    ["uv", "pip", "install", "-q", "--system", "--torch-backend=cu128", "vllm"],
    check=True,
)
cf = "/content/cloudflared"
if not Path(cf).exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf,
    )
    subprocess.run(["chmod", "+x", cf], check=True)

vllm_cmd = [
    "vllm",
    "serve",
    BASE_MODEL,
    "--port",
    str(PORT),
    "--quantization",
    "bitsandbytes",
    "--load-format",
    "bitsandbytes",
    "--gpu-memory-utilization",
    "0.92",
    "--max-model-len",
    "12288",
    "--enable-lora",
    "--max-lora-rank",
    "16",
    "--lora-modules",
    f"sqltuned={adapter_dir}",
]
print(" ".join(vllm_cmd), flush=True)
vllm_log = open("/content/vllm.log", "w")  # noqa: SIM115 - handle is the child's stdout, must outlive this cell
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

print("waiting for vllm (первый старт качает ~5.5GB весов)...", flush=True)
for _ in range(240):  # up to 40 min
    time.sleep(10)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5)
        break
    except Exception:
        if vllm_proc.poll() is not None:
            print(Path("/content/vllm.log").read_text()[-4000:])
            raise RuntimeError("vllm died during startup") from None
else:
    raise RuntimeError("vllm never became ready")

# smoke BOTH served names before exposing the URL
for name in ("sqltuned", BASE_MODEL):
    body = json.dumps(
        {
            "model": name,
            "messages": [{"role": "user", "content": "Reply with the word ok."}],
            "max_tokens": 5,
        }
    ).encode()
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.loads(r.read())["choices"][0]["message"]["content"]
    print(f"smoke {name}: {out!r}", flush=True)

tun = subprocess.Popen(
    [cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
assert tun.stdout is not None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print(f"\n{'=' * 60}\nTUNNEL: {url}\nbase_url for .env: {url}/v1\n{'=' * 60}\n", flush=True)

# push the URL out through the private nlsql-tunnel dataset (Windows polls it)
relay = Path("/content/relay")
relay.mkdir(exist_ok=True)
(relay / "tunnel_url.txt").write_text(url or "NONE")
(relay / "dataset-metadata.json").write_text(
    json.dumps(
        {"title": "nlsql-tunnel", "id": "liovinajo/nlsql-tunnel", "licenses": [{"name": "CC0-1.0"}]}
    )
)
for verb in (["version", "-m", "tunnel-url"], ["create"]):
    rr = subprocess.run(
        ["kaggle", "datasets", *verb, "-p", str(relay)], capture_output=True, text=True
    )
    print(f"relay {verb[0]}: rc={rr.returncode}", flush=True)
    if rr.returncode == 0:
        break
else:
    print("relay FAILED; copy the URL above manually", flush=True)

deadline = time.time() + SERVE_HOURS * 3600
while time.time() < deadline and vllm_proc.poll() is None:
    time.sleep(300)
    print(f"alive, url={url}, {int((deadline - time.time()) / 60)} min left", flush=True)
print("serve window over")